In [2]:
import requests
import json
import csv
import time
import logging
from datetime import datetime
from pathlib import Path

# --- Logging ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(f"pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
    ]
)
log = logging.getLogger(__name__)

# --- Config ---
OLLAMA_URL     = "http://localhost:11434/api/chat"
QUESTIONS_FILE = "questions.json"
OUTPUT_DIR     = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

MODELS = [
    "qwen2.5:7b",
    "gemma2:9b",
    "llama3.1:8b",
]

# Number of times to run the full pipeline
N_RUNS = 5

# Temperature for model responses (0.0 = deterministic, 1.0 = max variance)
TEMPERATURE = 0.8

SYSTEM_PROMPT = """
You are a financial decision maker evaluating investment choices.
For multiple choice questions, respond in this exact format:
ANSWER: [A or B]
RATIONALE: [exactly 3 sentences explaining your reasoning]

For open-ended questions, respond in this exact format:
RATIONALE: [exactly 3 sentences explaining your overall reasoning]

Stay consistent with your decision-making style throughout.
Do not add anything outside of the format above.
""".strip()

FIELDNAMES = [
    "run_id", "model", "section", "question_id", "pair",
    "frame", "question", "answer", "rationale", "raw_response"
]


In [4]:

# --- Ollama call with retry ---
def ask(model: str, prompt: str, retries: int = 3, delay: float = 2.0) -> str:
    """Send a single-turn prompt to Ollama. Retries on failure or empty response."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                OLLAMA_URL,
                json={
                    "model": model,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt},
                    ],
                    "stream": False,
                    "options": {
                        "temperature": TEMPERATURE,
                    },
                },
                timeout=120,
            )
            response.raise_for_status()
            content = response.json()["message"]["content"].strip()
            if content:
                return content
            log.warning(f"[{model}] Empty response on attempt {attempt}")
        except Exception as e:
            log.warning(f"[{model}] Attempt {attempt} failed: {e}")
        time.sleep(delay)

    log.error(f"[{model}] All {retries} attempts failed. Returning empty string.")
    return ""


# --- Response parser ---
def parse_response(raw: str, is_open_ended: bool = False):
    """Extract ANSWER and RATIONALE from model output."""
    answer    = ""
    rationale = ""

    for line in raw.strip().splitlines():
        if not is_open_ended and line.startswith("ANSWER:"):
            raw_answer = line.replace("ANSWER:", "").strip()
            answer = raw_answer[0].upper() if raw_answer else ""
        elif line.startswith("RATIONALE:"):
            rationale = line.replace("RATIONALE:", "").strip()

    if not answer and not is_open_ended:
        log.warning(f"Could not parse ANSWER from: {raw[:120]}")
    if not rationale:
        log.warning(f"Could not parse RATIONALE from: {raw[:120]}")

    return answer, rationale


# --- Prompt builder ---
def format_question(q: dict, instruction: str) -> str:
    """Build the full prompt string for a question."""
    prompt = q["prompt"]
    if q["choices"]:
        choices_text = "\n".join(f"{k}) {v}" for k, v in q["choices"].items())
        return f"{prompt}\n\n{choices_text}\n\n{instruction}"
    return f"{prompt}\n\n{instruction}"


# --- Unload model from VRAM ---
def unload_model(model: str):
    """Tell Ollama to evict the model from VRAM immediately."""
    try:
        requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "keep_alive": 0},
            timeout=10,
        )
        log.info(f"Unloaded {model} from VRAM")
        time.sleep(3)  # brief pause to let VRAM free up
    except Exception as e:
        log.warning(f"Could not unload {model}: {e}")


# --- CSV writer helper ---
def save_results(results: list[dict], model: str, run_id: int):
    """Save per-model per-run results to its own CSV file."""
    safe_model = model.replace(":", "_").replace("/", "_")
    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    path       = OUTPUT_DIR / f"results_{safe_model}_run{run_id:02d}_{timestamp}.csv"

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(results)

    log.info(f"Saved {len(results)} rows → {path}")
    return path


# --- Run one model N times ---
def run_model(data: dict, model: str) -> list[dict]:
    """Run a single model N_RUNS times over all questions. Unloads after."""
    all_model_results = []

    for run_id in range(1, N_RUNS + 1):
        log.info(f"\n{'='*60}")
        log.info(f"Model: {model} | Run {run_id}/{N_RUNS}")
        log.info(f"{'='*60}")

        run_results = []

        for section in data["sections"]:
            section_id  = section["section"]
            instruction = section["instruction"]

            log.info(f"\n--- Section: {section_id} ---")

            for q in section["questions"]:
                is_open_ended = q["choices"] is None
                prompt        = format_question(q, instruction)

                log.info(f"[{q['id']}] {q['prompt'][:80]}")

                raw               = ask(model, prompt)
                answer, rationale = parse_response(raw, is_open_ended)

                log.info(f"  ANSWER:    {answer or '(open-ended)'}")
                log.info(f"  RATIONALE: {rationale[:80]}...")

                row = {
                    "run_id":       run_id,
                    "model":        model,
                    "section":      section_id,
                    "question_id":  q["id"],
                    "pair":         q.get("pair", ""),
                    "frame":        q.get("frame", ""),
                    "question":     q["prompt"],
                    "answer":       answer,
                    "rationale":    rationale,
                    "raw_response": raw,
                }
                run_results.append(row)

        save_results(run_results, model, run_id)
        all_model_results.extend(run_results)

    unload_model(model)
    return all_model_results


# --- Main pipeline ---
def run_pipeline():
    with open(QUESTIONS_FILE, encoding="utf-8") as f:
        data = json.load(f)

    all_results = []

    for model in MODELS:
        log.info(f"\n{'#'*60}")
        log.info(f"STARTING MODEL: {model} ({N_RUNS} runs)")
        log.info(f"{'#'*60}")
        model_results = run_model(data, model)
        all_results.extend(model_results)

    # Save combined CSV with everything
    combined_path = OUTPUT_DIR / f"results_ALL_{N_RUNS}runs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    with open(combined_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(all_results)

    log.info(f"\n✓ Pipeline complete.")
    log.info(f"  Models run:        {len(MODELS)}")
    log.info(f"  Runs per model:    {N_RUNS}")
    log.info(f"  Total rows saved:  {len(all_results)}")
    log.info(f"  Combined CSV:      {combined_path}")


In [5]:
run_pipeline()

2026-03-13 01:37:30,006 [INFO] 
############################################################
2026-03-13 01:37:30,011 [INFO] STARTING MODEL: qwen2.5:7b (5 runs)
2026-03-13 01:37:30,012 [INFO] ############################################################
2026-03-13 01:37:30,013 [INFO] 
2026-03-13 01:37:30,013 [INFO] Model: qwen2.5:7b | Run 1/5
2026-03-13 01:37:30,015 [INFO] ============================================================
2026-03-13 01:37:30,015 [INFO] 
--- Section: H1 ---
2026-03-13 01:37:30,015 [INFO] [H1_Q1] Which do you prefer?
2026-03-13 01:37:32,913 [INFO]   ANSWER:    A
2026-03-13 01:37:32,914 [INFO]   RATIONALE: The guaranteed gain of $25 provides a certain outcome, avoiding the risk associa...
2026-03-13 01:37:32,914 [INFO] [H1_Q2] Which do you prefer?
2026-03-13 01:37:33,583 [INFO]   ANSWER:    A
2026-03-13 01:37:33,583 [INFO]   RATIONALE: The guaranteed gain provides a certain $200, avoiding the risk associated with t...
2026-03-13 01:37:33,584 [INFO] [H1_Q3] Which 